# 웹 문서 요약 with RAG — Map-Reduce 패턴으로 긴 문서 처리

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Map-Reduce** 패턴으로 LLM 컨텍스트 한계를 초과하는 문서 요약하기
- 커스텀 프롬프트로 Map 단계(청크 요약)와 Reduce 단계(통합 요약)를 각각 설정하기
- 요약본과 원본 청크를 함께 벡터스토어에 저장하는 하이브리드 RAG 구축하기

## 사전 지식

- 01-RAG-Web-Based.ipynb의 웹 문서 크롤링 방법
- LangChain의 `|` 연산자로 체인 연결하는 방법

---

```mermaid
flowchart TD
    WEB[웹 페이지]:::input --> C[청크 분할<br/>3000자]:::process
    subgraph MAP[Map 단계]
        C --> M1[청크1 요약]:::process
        C --> M2[청크2 요약]:::process
        C --> M3[청크3 요약]:::process
    end
    M1 --> R[Reduce 단계<br/>통합 요약]:::process
    M2 --> R
    M3 --> R
    R --> VS[(벡터스토어<br/>원본+요약)]:::storage
    Q[사용자 질문]:::input --> VS
    VS --> A[정확한 답변]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## Map-Reduce 요약 전략

| 단계 | 역할 | 입력 | 출력 |
|------|------|------|------|
| **Map** | 각 청크를 독립적으로 요약 | 청크 하나 | 짧은 요약 |
| **Reduce** | 모든 요약을 하나로 통합 | 모든 Map 결과 | 최종 요약 |

> **실무 팁**: `chunk_size=3000`처럼 크게 설정하면 Map 단계 횟수가 줄어 API 비용이 절약돼요. 단, 청크가 너무 크면 LLM이 핵심을 놓칠 수 있으니 1000~3000자 범위가 적당해요.

## 왜 요약이 필요한가?

### 문제 상황

- 웹 페이지는 수만 자의 긴 텍스트 포함
- 모든 내용을 검색 대상으로 하면 노이즈 증가
- LLM 컨텍스트 윈도우 제한

### 해결책: 요약 기반 RAG

```
긴 웹 문서 (10,000자)
    ↓
자동 요약 (2,000자)
    ↓
벡터 검색 (효율적)
    ↓
정확한 답변
```

In [1]:
import os
from dotenv import load_dotenv

# 아래에 코드를 작성하세요
load_dotenv()

True

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser


USER_AGENT environment variable not set, consider setting it to identify your requests.



# 요약을 위한 분할 (청크 크기를 크게)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,  # 요약용으로 큰 청크
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(docs)

print(f"분할된 청크 수: {len(split_docs)}")
if split_docs:
    print(f"첫 번째 청크 길이: {len(split_docs[0].page_content)} 문자")
else:
    print("⚠️ 충분한 텍스트를 가져오지 못해 청크가 생성되지 않았습니다.")

In [ ]:
# ---------------------------------------------------
# WebBaseLoader로 Wikipedia 딥러닝 페이지 크롤링
# ---------------------------------------------------
# ============================================================
# TODO: WebBaseLoader와 SoupStrainer로 Wikipedia 딥러닝 페이지를 로드하세요
# 힌트: bs_kwargs={"parse_only": bs4.SoupStrainer("div", attrs={"class": "mw-parser-output"})}
# 예상 결과: 문서 수, 총 문자 수 출력
# ============================================================

# Wikipedia 긴 문서 로드
# SoupStrainer: 본문 영역만 추출하여 네비게이션 등 HTML 노이즈 제거

loader = WebBaseLoader(
    web_paths=("https://ko.wikipedia.org/wiki/딥_러닝",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            "div",
            attrs={"id": "bodyContent"}
        )
    ),
)

docs = loader.load()

print(f' ==> [Line 17]: \033[38;2;226;237;24m[docs]\033[0m({type(docs).__name__}) = \033[38;2;164;22;104m{docs}\033[0m')


## 2. 문서 분할 (요약 준비)

In [6]:
# ---------------------------------------------------
# 문서를 Map-Reduce 요약에 적합한 크기로 분할
# ---------------------------------------------------
# ============================================================
# TODO: chunk_size=3000으로 문서를 분할하세요
# 힌트: RecursiveCharacterTextSplitter(chunk_size=3000, chunk_overlap=200)
# 예상 결과: 분할된 청크 수와 첫 청크 길이 출력
# ============================================================

# 요약용 청크 — chunk_size를 크게 설정해 Map 단계 횟수와 API 비용 절약
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(docs)
len(split_docs)

14

In [7]:

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Map 프롬프트 — 각 청크를 독립적으로 요약
base_map_prompt = PromptTemplate.from_template("""
다음 텍스트를 2~3문장으로 요약하세요.
텍스트: {text}

요약:
""")

# Reduce 프롬프트 — Map 결과를 하나로 통합
base_reduce_prompt = PromptTemplate.from_template("""
다음은 문서의 각 부분의 요약입니다.
이 요약을 바탕으로 전체 문서를 대표하는 요약을 5문장 이내로 작성하세요.

부분 요약들:
{text}

최종요약
""")

map_chain = base_map_prompt | llm | StrOutputParser()
reduce_chain = base_reduce_prompt | llm | StrOutputParser()

# 💡 팁: map_chain.batch([{"text": d.page_content} for d in split_docs])로 병렬화 가능
# Reduce 단계: 모든 요약 통합 (LLM 호출 1회)
# ⚠️ 주의: map_summaries가 너무 많으면 Reduce 단계도 컨텍스트 초과 가능 → 계층적 Reduce 고려
map_summaries = [map_chain.invoke({"text": doc.page_content}) for doc in split_docs]
print(f' ==> [Line 29]: \033[38;2;174;80;22m[map_summaries]\033[0m({type(map_summaries).__name__}) = \033[38;2;100;17;190m{map_summaries}\033[0m')

summary = reduce_chain.invoke({"text": "\n".join(map_summaries)})




 ==> [Line 29]: [map_summaries](list) = ['위키백과의 기계 학습과 데이터 마이닝 관련 내용은 다양한 학습 패러다임(지도 학습, 비지도 학습 등)과 문제 유형(분류, 회귀, 군집화 등), 그리고 여러 알고리즘(인공 신경망, 강화 학습 등)을 포함하고 있다. 또한, 기계 학습의 응용 분야와 관련된 철학적 논의, 역사적 배경, 그리고 논란에 대해서도 다루고 있다.', '딥 러닝은 비선형 변환 기법을 통해 데이터에서 높은 수준의 추상화를 시도하는 기계 학습 알고리즘의 집합으로, 컴퓨터 비전, 음성 인식, 자연어 처리 등 다양한 분야에 적용되고 있다. 이 기술은 인공신경망의 발전과 함께 하드웨어의 개선, 빅 데이터의 활용 덕분에 최근에 다시 주목받고 있으며, 과거의 문제점들이 해결되면서 더욱 효과적으로 발전하고 있다.', '빅 데이터는 SNS 사용자들이 생성한 대량의 자료와 태그 정보를 수집하고 분석하여 인공신경망의 학습에 활용된다. 딥 러닝의 발전으로 자동 음성 인식과 컴퓨터 비전 분야에서 뛰어난 성능을 보여주며, 다양한 심층 신경망 구조가 복잡한 비선형 관계를 모델링할 수 있도록 돕는다. 특히, 합성곱 신경망(CNN)은 음향 모델링과 같은 분야에서 기존 모델보다 더 성공적으로 적용되고 있다.', '주어진 텍스트는 심층 신경망의 학습 과정과 관련된 수식 및 개념을 설명하고 있습니다. 학습률(η)과 비용함수(C)의 선택은 학습 형태와 활성화 함수에 따라 달라지며, 예를 들어 다종 분류 문제에서는 softmax 함수와 교차 엔트로피 함수가 일반적으로 사용됩니다. 또한, 심층 신경망은 과적합과 높은 시간 복잡도와 같은 문제에 취약하며, 이를 해결하기 위해 weight decay와 sparsity 같은 기법이 필요하다고 언급하고 있습니다.', '정칙화 방법 중 하나인 dropout은 심층 신경망의 학습 과정에서 일부 유닛을 임의로 생략하여 rare dependency 문제를 해결하는 데 도움을 준다. 합성곱 신경망(CNN)은 최소한의 전처리로

In [8]:
summary

'이 문서는 기계 학습과 딥 러닝의 다양한 패러다임, 알고리즘, 응용 분야를 포괄적으로 다루고 있다. 딥 러닝은 인공신경망의 발전과 빅 데이터의 활용 덕분에 최근 크게 발전하였으며, 음성 인식, 컴퓨터 비전, 자연어 처리 등에서 뛰어난 성능을 보여준다. 심층 신경망의 학습 과정과 관련된 기술적 세부사항, 정칙화 방법, 그리고 가중치 업데이트 과정도 설명되어 있다. 또한, 딥 러닝의 이론적 기반과 그에 대한 비판, 다양한 산업에서의 응용 사례를 통해 기계 학습의 중요성을 강조하고 있다. 이 기술은 인공지능의 발전에 기여하며, 앞으로의 가능성을 열어주는 핵심 기술로 자리잡고 있다.'

In [ ]:
# ---------------------------------------------------
# Map-Reduce 패턴으로 긴 문서 요약
# ---------------------------------------------------
# ============================================================
# TODO: map_chain으로 각 청크를 요약(Map)하고, reduce_chain으로 통합(Reduce)하세요
# 힌트: map_summaries = [map_chain.invoke({"text": doc.page_content}) for doc in split_docs]
# 예상 결과: 전체 문서 요약 출력 및 압축률 표시
# ============================================================


    # Map 프롬프트 — 각 청크를 독립적으로 요약


    # Reduce 프롬프트 — Map 결과를 하나로 통합



    # Map 체인과 Reduce 체인 구성

    # Map 단계: 각 청크 독립 요약

    # Reduce 단계: 모든 요약 통합


## 4. 맞춤 요약 프롬프트

기본 요약 대신, 특정 관점으로 요약할 수 있습니다.

In [9]:
# ---------------------------------------------------
# 커스텀 Map/Reduce 프롬프트로 핵심 내용 위주 요약
# ---------------------------------------------------
# ============================================================
# TODO: 커스텀 map_prompt와 reduce_prompt를 정의하고 요약 파이프라인을 실행하세요
# 힌트: custom_map_chain = map_prompt | llm | StrOutputParser()
# 예상 결과: 커스텀 요약 결과 출력
# ============================================================
map_prompt = PromptTemplate.from_template(
    """다음 텍스트를 **핵심 내용 위주**로 간결하게 요약하세요.
중요한 개념, 정의, 예시만 포함하세요.

텍스트:
{text}

간결한 요약:"""
)

# 커스텀 Reduce 프롬프트 — 일관된 통합 요약
reduce_prompt = PromptTemplate.from_template(
    """다음은 문서의 여러 부분을 요약한 내용입니다.
이들을 하나의 일관된 요약으로 통합하세요.

부분 요약들:
{text}

통합 요약:"""
)


custom_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
custom_map_chain = map_prompt | custom_llm | StrOutputParser()
custom_reduce_chain = reduce_prompt | custom_llm | StrOutputParser()
partial_summaries = [custom_map_chain.invoke({"text": doc.page_content}) for doc in split_docs]
custom_summary = custom_reduce_chain.invoke({"text": "\n".join(partial_summaries)})

print(":memo: 커스텀 요약:")
print("=" * 60)
print(custom_summary)


:memo: 커스텀 요약:
기계 학습과 데이터 마이닝은 다양한 학습 패러다임과 기법을 통해 데이터에서 유용한 정보를 추출하는 분야입니다. 주요 학습 방식으로는 지도 학습, 비지도 학습, 준지도 학습, 자기 지도 학습, 강화 학습, 메타 학습 등이 있으며, 문제 유형으로는 분류, 회귀, 군집화, 차원 축소, 이상 탐지가 있습니다. 지도 학습 기법으로는 결정 트리, 앙상블 기법, 인공 신경망, 서포트 벡터 머신(SVM) 등이 있으며, 군집화 기법으로는 k-평균, DBSCAN, 계층적 군집화가 있습니다. 차원 축소 기법으로는 PCA, t-SNE, LDA 등이 사용됩니다.

딥 러닝은 기계 학습의 한 분야로, 인공 신경망(ANN)을 기반으로 하여 데이터의 높은 수준의 추상화를 시도합니다. 딥 뉴럴 네트워크(DNN), 합성곱 신경망(CNN), 순환 신경망(RNN) 등 다양한 기법이 있으며, 이들은 이미지, 음성, 자연어 처리 등 여러 분야에 적용됩니다. 딥 러닝의 발전은 GPU와 빅 데이터의 활용 증가, 기존 모델의 한계 극복 등으로 인해 가속화되었습니다.

딥 러닝의 주요 개념으로는 표현(representation) 학습, 과적합(overfitting) 문제, 정칙화(Regularization) 기법 등이 있으며, 학습 방법으로는 오류역전파 알고리즘과 확률적 경사 하강법이 사용됩니다. 특히, 합성곱 신경망(CNN)은 컴퓨터 비전과 음성 인식 분야에서 뛰어난 성능을 보이며, 순환 신경망(RNN)은 시퀀스 데이터 처리에 강점을 가지고 있습니다.

딥 러닝은 자동 음성 인식, 이미지 분류, 자연어 처리, 약물 발견 등 다양한 응용 분야에서 성공적으로 활용되고 있으며, 이론적 뒷받침이 부족하다는 비판도 존재합니다. 주요 연구와 기법으로는 제한 볼츠만 머신(RBM), 심층 신뢰 신경망(DBN), 심층 Q-네트워크(DQN) 등이 있으며, TensorFlow, Caffe, Theano와 같은 소프트웨어 라이브러리가 널리 사용됩니다.

마지막으로, 기계 학습과 딥 러닝의 발전은 A

In [ ]:
# ---------------------------------------------------
# 원본 청크 + 요약 문서를 통합한 하이브리드 RAG 구축
# ---------------------------------------------------
# ============================================================
# TODO: 원본 split_docs와 요약 summary_doc을 합쳐 FAISS 벡터스토어와 RAG 체인을 만드세요
# 힌트: all_docs = split_docs + [summary_doc], FAISS.from_documents(all_docs, embeddings)
# 예상 결과: "요약 기반 RAG 시스템 구축 완료" 출력
# ============================================================

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document


# 요약을 Document 객체로 변환 (source="summary"로 메타데이터 구분)
summary_doc = Document(
    page_content=custom_summary,
    metadata={"source": "summary", "url": "https://ko.wikipedia.org/wiki/딥_러닝"}
)

# 원본 문서 + 요약 함께 저장 — 세부 질문은 원본에서, 개요 질문은 요약에서 검색
all_docs = split_docs + [summary_doc]

# 벡터스토어 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(all_docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG 체인
rag_prompt = PromptTemplate.from_template(
"""문맥을 참고하여 질문에 답하세요.

문맥:
{context}

질문: {question}

답변:"""
)
rag_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | rag_prompt
    | rag_llm
    | StrOutputParser()
)

print(":white_check_mark: 요약 기반 RAG 시스템 구축 완료")
print(f"- 원본 청크: {len(split_docs)}개")
print(f"- 요약 문서: 1개")
print(f"- 총 검색 대상: {len(all_docs)}개")

:white_check_mark: 요약 기반 RAG 시스템 구축 완료
- 원본 청크: 14개
- 요약 문서: 1개
- 총 검색 대상: 15개


## 6. 질의응답 테스트

In [ ]:
# 아래에 코드를 작성하세요
question = "요약 내용을 기반으로 주요 활용 사례를 알려주세요."
answer = rag_chain.invoke(question)

In [13]:
# 아래에 코드를 작성하세요

print(f' ==> [Line 2]: \033[38;2;155;171;110m[answer]\033[0m({type(answer).__name__}) = \033[38;2;243;108;82m{answer}\033[0m')


 ==> [Line 2]: [answer](str) = 딥 러닝의 주요 활용 사례는 다음과 같습니다:

1. **자동 음성 인식**: 딥 러닝 기법이 모든 주요 상업 음성 인식 시스템(예: MS 코타나, 구글 나우, 애플 시리 등)에 적용되고 있으며, 대량 어휘 음성 인식 분야에서 성공적인 사례로 자리잡고 있습니다.

2. **영상 인식**: 이미지 분류 및 사물 인식 분야에서 딥 러닝이 중요한 역할을 하고 있으며, 특히 대규모 ImageNet 대회에서의 성과로 그 중요성이 부각되었습니다. 또한, 자동 영상 캡셔닝과 같은 도전적인 분야로의 확장도 이루어지고 있습니다.

3. **자연어 처리**: 딥 러닝은 감정 분석, 정보 검색 등 다양한 자연어 처리 관련 연구에서 최첨단 기술로 사용되고 있으며, 단어 표현 및 재귀 신경망을 활용하여 언어 모델을 구현하는 데 기여하고 있습니다.

4. **약물 발견과 독성학**: 제약 산업에서 딥 러닝을 활용하여 약의 효능을 예측하고 예상치 못한 작용을 사전에 탐지하는 데 성공적인 사례가 있습니다.

5. **고객 관계 관리**: 딥 러닝을 활용하여 고객 생명 주기 값을 예측하고, 마케팅 활동의 가치를 평가하는 데 사용되고 있습니다.

이 외에도 다양한 분야에서 딥 러닝 기술이 적용되고 있으며, 그 활용 가능성은 계속해서 확장되고 있습니다.


## 💡 핵심 정리

### Map-Reduce 요약의 장점

1. **긴 문서 처리**: LLM 컨텍스트 제한 극복
2. **병렬 처리**: 각 청크를 독립적으로 요약 (속도 향상 가능)
3. **유연성**: Map/Reduce 프롬프트 커스터마이징

### 요약 체인 타입

| 타입 | 설명 | 장점 | 단점 |
|------|------|------|------|
| **stuff** | 모든 문서를 한 번에 | 빠름 | 긴 문서 불가 |
| **map_reduce** | 청크별 요약 후 통합 | 긴 문서 가능 | 느림 |
| **refine** | 순차적 정제 요약 | 일관성 높음 | 매우 느림 |

### 실전 활용

- 뉴스 기사 요약 및 QA
- 논문/보고서 분석
- 웹 콘텐츠 요약 봇
- 긴 회의록 요약

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 패턴 | Map(청크별 독립 요약) → Reduce(전체 통합 요약) |
| 저장 전략 | 원본 청크 + 요약 청크 모두 VectorStore에 통합 |
| Map 단계 | 각 청크를 독립적으로 요약 — 병렬 처리 가능 |
| Reduce 단계 | Map 결과를 합쳐 최종 요약문 생성 |
| 장점 | LLM 컨텍스트 제한을 넘어서는 긴 문서 처리 가능 |

### Map-Reduce vs 단순 요약

| 방식 | 처리 가능 길이 | 일관성 | LLM 호출 수 |
|------|----------------|--------|-------------|
| 단순 요약 | 컨텍스트 내 | 높음 | 1회 |
| Map-Reduce | 무제한 | 중간 | 청크 수 + 1 |
| Refine (점진) | 무제한 | 높음 | 청크 수 × 2 |

### 6-7 RAG Process 시리즈 완성

| 노트북 | 핵심 기술 | 적합한 상황 |
|--------|-----------|-------------|
| 00-RAG-Basic | 8단계 기본 파이프라인 | RAG 처음 시작할 때 |
| 01-Web-Based | WebBaseLoader + BeautifulSoup | 웹 문서 실시간 QA |
| 02-Advanced | Ensemble + MMR | 검색 다양성이 필요할 때 |
| 03-Conversation | MessageHistory + 세션 | 멀티턴 대화 시스템 |
| 04-RAPTOR | 계층적 요약 인덱스 | 매우 긴 문서 전체 이해 |
| **05-Web-Summarization** | **Map-Reduce 요약** | **웹 문서 대규모 요약 + QA** |

6-5 Retriever부터 6-7 RAG Process까지 모든 핵심 RAG 기법을 학습했어요. 이제 실제 프로젝트에 다양한 기법을 조합해 적용해 보세요!